# LLM-Only Multi-Agent Experiment

This notebook verifies the isolated experiment, runs one retained benchmark example, and optionally scores the result. Human references remain unavailable to every generation stage and are used only after generation by the evaluator.

## 1. Environment

Resolve the repository paths, expose the main evaluation package and experiment package, then load the project-level `.env`.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in search_roots if (path / "table2text_pydanticai").exists())
MAIN_PROJECT = PROJECT_ROOT / "table2text_pydanticai"
EXPERIMENT_ROOT = PROJECT_ROOT / "experiments/llm_only_pipeline"

for source_root in [MAIN_PROJECT / "src", EXPERIMENT_ROOT / "src"]:
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

from table2text.evaluation import (  # noqa: E402
    generate_reports_for_notebook,
    load_project_env,
    score_reference_metrics_for_notebook,
)
from table2text.evaluation.datasets import read_examples  # noqa: E402
from table2text_llm_only.workflow import LLMOnlyWorkflow  # noqa: E402

load_project_env(MAIN_PROJECT)
print("Project:", PROJECT_ROOT)
print("Experiment:", EXPERIMENT_ROOT)
print("API key available:", bool(os.getenv("DEEPSEEK_API_KEY")))

## 2. Run Configuration

Choose `llm_only_flash` for the primary experiment or `llm_only_pro` for the optional model-strength comparison. API and metric execution are off by default.

In [ ]:
VARIANT_ID = "llm_only_flash"
RUN_LIVE_GENERATION = False
RUN_REFERENCE_METRICS = False
RUN_SOURCE_GROUNDED_METRICS = False

EXAMPLES_PATH = EXPERIMENT_ROOT / "data/sportsett_basketball_4934.jsonl"
VARIANTS_PATH = EXPERIMENT_ROOT / "config/variants.json"
RUN_ROOT = EXPERIMENT_ROOT / "artifacts/runs" / VARIANT_ID
GENERATIONS_PATH = RUN_ROOT / "generations.jsonl"

print("Variant:", VARIANT_ID)
print("Example file:", EXAMPLES_PATH)

## 3. Reference-Isolation Check

The benchmark record retains references for later scoring. This check confirms that the workflow source packet excludes them.

In [ ]:
examples = read_examples(EXAMPLES_PATH)
assert len(examples) == 1
example = examples[0]

workflow_without_client = object.__new__(LLMOnlyWorkflow)
workflow_without_client.max_source_characters = 100_000
workflow_without_client.max_source_payload_characters = 5_000
packet = workflow_without_client._source_packet(example)

assert "references" not in packet
assert "reference_sha256" not in packet
print(f"Loaded {example.dataset_id}/{example.example_id}")
print("Reference isolation: PASS")
print("Source characters:", len(example.source_text))

## 4. Generate

This cell creates a one-variant config, reports progress, and writes both the evaluator record and detailed agent artifact.

In [ ]:
generation_frame = None

if RUN_LIVE_GENERATION:
    payload = json.loads(VARIANTS_PATH.read_text(encoding="utf-8"))
    selected = [item for item in payload["variants"] if item["variant_id"] == VARIANT_ID]
    if not selected:
        raise ValueError(f"Unknown variant: {VARIANT_ID}")
    selected[0]["enabled"] = True

    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    selected_config = RUN_ROOT / "variant.json"
    selected_config.write_text(json.dumps({"variants": selected}, indent=2), encoding="utf-8")

    print(f"Starting {VARIANT_ID} on {example.dataset_id}/{example.example_id}...")
    started = time.perf_counter()
    generation_frame = await generate_reports_for_notebook(
        MAIN_PROJECT,
        examples_path=EXAMPLES_PATH,
        variants_path=selected_config,
        output_path=GENERATIONS_PATH,
        run_root=RUN_ROOT / "pipeline",
        resume=False,
    )
    print(f"Generation complete in {time.perf_counter() - started:.1f}s")
else:
    print("Live generation skipped. Set RUN_LIVE_GENERATION = True to run it.")

## 5. Inspect Output

Show the final text and key execution diagnostics without exposing the held-out references.

In [ ]:
if generation_frame is not None:
    row = generation_frame.iloc[0]
    print("Error:", row.get("error"))
    print("Writer mode:", row.get("writer_mode"))
    print("Release status:", row.get("release_status"))
    print("Total tokens:", row.get("total_tokens"))
    print("\nGenerated text:\n")
    print(row.get("generated_text") or "<empty>")
else:
    print("No live result in memory.")

## 6. Optional Metrics

Reference-similarity metrics compare against held-out references. Source-grounded metrics use the structured source as context. These calls score an existing generation and do not rerun the agents.

In [ ]:
if RUN_REFERENCE_METRICS or RUN_SOURCE_GROUNDED_METRICS:
    if not GENERATIONS_PATH.exists():
        raise FileNotFoundError(f"Run generation first: {GENERATIONS_PATH}")

    RESULTS_DIR = RUN_ROOT / "metrics"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    if RUN_REFERENCE_METRICS:
        print("Scoring reference similarity...")
        reference_scores = score_reference_metrics_for_notebook(
            MAIN_PROJECT,
            generations_path=GENERATIONS_PATH,
            metric_config_path=MAIN_PROJECT / "evaluation/config/metrics_reference_similarity.json",
            output_path=RESULTS_DIR / "reference_metrics.jsonl",
            include_ineligible=True,
        )
        display(reference_scores[["metric_name", "status", "score"]])

    if RUN_SOURCE_GROUNDED_METRICS:
        print("Scoring source grounding...")
        source_scores = score_reference_metrics_for_notebook(
            MAIN_PROJECT,
            generations_path=GENERATIONS_PATH,
            metric_config_path=MAIN_PROJECT / "evaluation/config/metrics_source_grounded.json",
            output_path=RESULTS_DIR / "source_grounded_metrics.jsonl",
            include_ineligible=True,
        )
        display(source_scores[["metric_name", "status", "score"]])
else:
    print("Metric scoring skipped. Enable one or both metric flags above.")

## Preserved Case-Study Evidence

The historical Flash and Pro outputs and their metric tables are retained under `artifacts/sportsett_4934/`. New runs are written under `artifacts/runs/` and remain separate from the frozen evidence.